# Makemore Part 1: Bigram Models

This notebook builds the simplest character-level language model: a **bigram model**.
We first count how often every pair of characters occurs, then we build the same model
as a tiny neural network trained with gradient descent. The two approaches end up doing
the same math, which is a great way to see why the neural-net machinery is so powerful.

## Learning objectives
- Build a character vocabulary and turn text into integer indexes.
- Estimate a bigram probability table from counts and understand why smoothing matters.
- Sample new words from a probability model.
- Cast the bigram problem as a one-layer neural network with a softmax output.
- Train the network with negative log-likelihood (cross-entropy) and backpropagation.

## Why this matters
Bigrams are the smallest non-trivial language model. They let us study the full pipeline—
data → model → loss → optimization—without the complexity of deep architectures.
Once you understand bigrams, MLPs, batch normalization, and WaveNet-style models are just
richer versions of the same ideas.

## Prerequisites
- Basic Python and NumPy/PyTorch tensors.
- What a probability distribution is (summing to 1, sampling).
- Derivatives and the chain rule (we will use PyTorch autograd).
- The `nnzero` package installed in this repo (we import helpers from it).


In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from nnzero.utils import build_vocab, build_dataset, split_dataset, load_names, set_seed

%matplotlib inline

# Reproducibility and data loading. The path is relative to the repo root.
set_seed(2147483647)
words = load_names('data/names.txt')
print(f'Loaded {len(words)} names')


## Quick-run mode for CI

When the environment variable `NNZERO_QUICK_RUN=1` is set, training loops run for only a small number of steps so that notebooks can be validated quickly. Students should leave this unset to train the full models.

In [ ]:
import os
QUICK_RUN = os.environ.get("NNZERO_QUICK_RUN", "0") == "1"
if QUICK_RUN:
    print("Quick-run mode enabled: training loops will use far fewer steps.")

In [ ]:
words[:10]

In [ ]:
len(words)

In [ ]:
min(len(w) for w in words)

In [ ]:
max(len(w) for w in words)

## Why build a vocabulary?

Neural networks work with numbers, not characters. We map every character to a unique
integer index. The special `.` token marks both the start and the end of a name, so the
model learns how names begin and end.

> **Tip:** You can also call `stoi, itos, vocab_size = build_vocab(words)` from `nnzero`.
> We build it by hand here so you see exactly how the `.` token is handled.


In [ ]:
b = {}
for w in words:
  chs = ['<S>'] + list(w) + ['<E>']
  for ch1, ch2 in zip(chs, chs[1:]):
    bigram = (ch1, ch2)
    b[bigram] = b.get(bigram, 0) + 1

In [ ]:
sorted(b.items(), key = lambda kv: -kv[1])

## Why a count matrix?

A 27 × 27 tensor `N` stores how many times character `i` is followed by character `j`.
Rows are the current character, columns are the next character. This compact table is
all we need to estimate bigram probabilities.


In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [ ]:

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1
    

In [ ]:
plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [ ]:
N[0]

## Why normalize rows into probabilities?

To generate a new name, we repeatedly ask: *given the current character, what character
comes next?* We answer by normalizing each row of `N` into a probability distribution
and then sampling from it with `torch.multinomial`.


In [ ]:
p = N[0].float()
p = p / p.sum()
p

In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos[ix]

In [ ]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p = p / p.sum()
p

In [ ]:
torch.multinomial(p, num_samples=100, replacement=True, generator=g)

In [ ]:
p.shape

Shape check:
- `N` has shape `(27, 27)`.
- `P.sum(1, keepdim=True)` has shape `(27, 1)` and broadcasts over columns.


If you forget `keepdim=True`, the result has shape `(27,)` instead of `(27, 1)`,
and broadcasting can silently give the wrong answer. Always check shapes!


## Why add-one smoothing?

Any bigram that never appeared in the training data would get probability 0.
Multiplying by 0 in the likelihood gives `-inf` log-likelihood, and sampling could
get stuck. Adding a small constant (`N + 1`) gives every bigram a tiny non-zero
probability—this is called **Laplace smoothing**.


In [ ]:
P = (N+1).float()
P /= P.sum(1, keepdims=True)

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  
  out = []
  ix = 0
  while True:
    p = P[ix]
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

## Why negative log-likelihood?

A good model assigns high probability to the real data. We therefore maximize the
product of probabilities (the likelihood). Because `log(a*b) = log(a) + log(b)`,
maximizing the log-likelihood turns a product into a sum and avoids numerical underflow.
Minimizing the **negative** log-likelihood is the same optimization problem, and it is
the standard loss function for classification.


In [ ]:
# GOAL: maximize likelihood of the data w.r.t. model parameters (statistical modeling)
# equivalent to maximizing the log likelihood (because log is monotonic)
# equivalent to minimizing the negative log likelihood
# equivalent to minimizing the average negative log likelihood

# log(a*b*c) = log(a) + log(b) + log(c)

In [ ]:
log_likelihood = 0.0
n = 0

for w in words:
#for w in ["andrejq"]:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    #print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

### Try it yourself: tune the smoothing constant

The code above uses `N + 1` before normalizing. What happens if you use `N + 0`
(no smoothing) or `N + 10`? Recompute the average negative log-likelihood and sample
a few names for each value. How does smoothing affect quality and diversity?


#### Solution

In [ ]:
for alpha in [0, 1, 10]:
    P = (N + alpha).float()
    P /= P.sum(1, keepdims=True)
    log_likelihood = 0.0
    n = 0
    for w in words:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            ix1, ix2 = stoi[ch1], stoi[ch2]
            log_likelihood += torch.log(P[ix1, ix2])
            n += 1
    print(f'alpha={alpha}, nll/n={-log_likelihood/n:.4f}')

    g = torch.Generator().manual_seed(2147483647)
    for _ in range(3):
        out = []
        ix = 0
        while True:
            ix = torch.multinomial(P[ix], 1, generator=g).item()
            out.append(itos[ix])
            if ix == 0:
                break
        print('  ', ''.join(out))


## From counts to a neural network

The count table worked, but it cannot scale to richer contexts. We now build the
same bigram model as a tiny neural network: one linear layer from a one-hot input
to 27 logits, followed by a softmax. With enough training, the network learns a
weight matrix `W` that is equivalent to the log-count table.


In [ ]:
# create the training set of bigrams (x,y)
xs, ys = [], []

for w in words[:1]:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    print(ch1, ch2)
    xs.append(ix1)
    ys.append(ix2)
    
xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs

In [ ]:
ys

In [ ]:
xenc = F.one_hot(xs, num_classes=27).float()
xenc

In [ ]:
xenc.shape

In [ ]:
plt.imshow(xenc)

In [ ]:
xenc.dtype

### Why one-hot encoding?

A one-hot vector has a single `1` at the index of the current character. When we
multiply it by `W`, the matrix product picks out exactly one row of `W`. That row
is the set of log-counts (logits) for the next character, which is exactly what
the count matrix represented.


In [ ]:
W = torch.randn((27, 1))
xenc @ W

In [ ]:
logits = xenc @ W # log-counts
counts = logits.exp() # equivalent N
probs = counts / counts.sum(1, keepdims=True)
probs

In [ ]:
probs[0]

In [ ]:
probs[0].shape

In [ ]:
probs[0].sum()

The one-hot matrix `(5, 27)` times `W` `(27, 27)` produces logits `(5, 27)`.

---

**Summary so far:** counts → probabilities → one-hot input → logits → softmax.

In [ ]:
xs

In [ ]:
ys

### Why softmax?

The network outputs arbitrary real numbers (logits). We need a probability
distribution: all values positive and summing to 1. `softmax(x) = exp(x) / sum(exp(x))`
does exactly that and is differentiable, so we can train with gradient descent.


In [ ]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g)

In [ ]:
xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# btw: the last 2 lines here are together called a 'softmax'

In [ ]:
probs.shape

In [ ]:

nlls = torch.zeros(5)
for i in range(5):
  # i-th bigram:
  x = xs[i].item() # input character index
  y = ys[i].item() # label character index
  print('--------')
  print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
  print('input to the neural net:', x)
  print('output probabilities from the neural net:', probs[i])
  print('label (actual next character):', y)
  p = probs[i, y]
  print('probability assigned by the net to the the correct character:', p.item())
  logp = torch.log(p)
  print('log likelihood:', logp.item())
  nll = -logp
  print('negative log likelihood:', nll.item())
  nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

### Try it yourself: vectorize the loss

The cell above computes the loss with a Python loop. Write a single expression
using `F.cross_entropy` that computes the same value from `logits` and `ys`.


#### Solution

In [ ]:
loss_vec = F.cross_entropy(logits, ys)
print(loss_vec.item())


## Why gradient descent?

We have a loss that tells us how wrong the model is. To make it less wrong, we compute
the gradient of the loss with respect to `W` and nudge `W` in the opposite direction.
PyTorch's autograd does the chain-rule bookkeeping for us.


### Optimization with autograd

In [ ]:
xs

In [ ]:
ys

In [ ]:
# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# forward pass
xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
loss = -probs[torch.arange(5), ys].log().mean()

In [ ]:
print(loss.item())

In [ ]:
# backward pass
W.grad = None # set to zero the gradient
loss.backward()

In [ ]:
W.data += -0.1 * W.grad

## Training on the full dataset

Now we repeat the forward/backward/update loop on every bigram in the dataset.
Instead of writing the dataset construction again, we reuse `build_dataset` from
`nnzero`. (Try peeking at `nnzero/utils.py` to see how it works under the hood.)


In [ ]:
# Build the full bigram dataset using the shared helper.
# block_size=1 means the context is just the previous character.
xs, ys = build_dataset(words, stoi, block_size=1)
num = xs.nelement()
print('number of examples: ', num)

# initialize the network
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)


In [ ]:
# gradient descent
for k in range(1):
  
  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

## Why sample from the model?

Training gives us a good weight matrix, but we still need to turn it into new names.
We feed the model the start token `.`, sample the next character from the output
probabilities, feed that character back in, and repeat until we sample `.` again.


In [ ]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  
  out = []
  ix = 0
  while True:
    
    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))